# M10 final tables and run manifest
Run the frozen robustness/reporting layer and synchronize the final handoff. No robustness winner is selected here.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/a0f8044c6dfb61c3ff029d4e92e2cac0327bfe44/scripts/colab_bootstrap.py'
urllib.request.urlretrieve(raw, str(_BOOTSTRAP_SCRIPT))
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Preflight all required upstream artifacts. Rebuild the derived tensor cache
# if an earlier runtime did not persist it to Drive.
import json, shutil, subprocess, sys
from shutil import copy2
baseline = WORKSPACE / 'runs' / 'v3_forecast_baselines'
portfolio = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks'
method = WORKSPACE / 'runs' / 'v3_ptcst_seed_sweep' / 'seed_7'
ablation = WORKSPACE / 'runs' / 'v3_ptcst_ablations'
stats = WORKSPACE / 'runs' / 'v3_statistical_tests'
risk = WORKSPACE / 'runs' / 'v3_risk_coverage'
cache = WORKSPACE / 'artifacts' / 'v3_tensor_cache'
if not (cache / 'cache_manifest.json').exists():
    print('Tensor cache is missing; rebuilding it before the reporting stage.')
    if cache.exists(): shutil.rmtree(cache)
    cmd = [sys.executable, str(REPO / 'scripts' / 'build_v3_tensor_cache.py'), '--data-root', str(DATA_ROOT), '--output-dir', str(cache)]
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:\n' + result.stderr)
    if result.returncode != 0: raise RuntimeError(f'Tensor cache rebuild failed: {result.stderr or result.stdout}')
    drive_cache = DRIVE_RUN_ROOT / 'artifacts' / 'v3_tensor_cache'
    shutil.copytree(cache, drive_cache, dirs_exist_ok=True)
    print('Rebuilt tensor cache and synced it to:', drive_cache)
leakage_report = WORKSPACE / 'runs' / 'v3_leakage_report.json'
if not leakage_report.exists():
    print('Leakage report is missing; rebuilding it from the frozen data.')
    cmd = [sys.executable, str(REPO / 'scripts' / 'validate_v3_leakage.py'), '--data-root', str(DATA_ROOT), '--report', str(leakage_report)]
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:\n' + result.stderr)
    if result.returncode != 0: raise RuntimeError(f'Leakage validation failed: {result.stderr or result.stdout}')
    copy2(leakage_report, DRIVE_RUN_ROOT / 'v3_leakage_report.json')
    print('Leakage report rebuilt and synced to Drive.')
determinism_report = WORKSPACE / 'runs' / 'v3_determinism_report.json'
if not determinism_report.exists() and (DRIVE_RUN_ROOT / 'v3_determinism_report.json').exists():
    copy2(DRIVE_RUN_ROOT / 'v3_determinism_report.json', determinism_report)
if not determinism_report.exists():
    print('Determinism report is missing; rebuilding the deterministic repeats.')
    baseline_repeat = WORKSPACE / 'runs' / 'v3_forecast_baselines_repeat'
    portfolio_repeat = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks_repeat'
    commands = [
        [sys.executable, str(REPO / 'scripts' / 'run_v3_forecast_baselines.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(baseline_repeat)],
        [sys.executable, str(REPO / 'scripts' / 'run_v3_portfolio_benchmarks.py'), '--data-root', str(DATA_ROOT), '--forecast-run', str(baseline_repeat), '--run-dir', str(portfolio_repeat)],
        [sys.executable, str(REPO / 'scripts' / 'validate_v3_determinism.py'), '--main-run', str(baseline), '--repeat-run', str(baseline_repeat), '--files', 'forecasts.parquet', 'forecast_metrics_by_date.parquet', 'forecast_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_forecast.json')],
        [sys.executable, str(REPO / 'scripts' / 'validate_v3_determinism.py'), '--main-run', str(portfolio), '--repeat-run', str(portfolio_repeat), '--files', 'portfolio_returns.parquet', 'weights.parquet', 'trades.parquet', 'solver_log.parquet', 'portfolio_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_portfolio.json')],
    ]
    for cmd in commands:
        result = subprocess.run(cmd, check=False, capture_output=True, text=True)
        if result.stdout: print(result.stdout)
        if result.stderr: print('STDERR:\n' + result.stderr)
        if result.returncode != 0: raise RuntimeError(f'Determinism rebuild failed: {result.stderr or result.stdout}')
    determinism = {'forecast': json.loads((WORKSPACE / 'runs' / 'v3_determinism_forecast.json').read_text()), 'portfolio': json.loads((WORKSPACE / 'runs' / 'v3_determinism_portfolio.json').read_text())}
    determinism_report.write_text(json.dumps(determinism, indent=2))
    copy2(determinism_report, DRIVE_RUN_ROOT / 'v3_determinism_report.json')
    print('Determinism report rebuilt and synced to Drive.')
required = [baseline / 'forecasts.parquet', portfolio / 'portfolio_metrics_summary.parquet', method / 'metrics.json', ablation / 'portfolio_metrics_summary.parquet', stats / 'statistical_tests.json', risk / 'metrics.json', cache / 'cache_manifest.json', WORKSPACE / 'runs' / 'v3_leakage_report.json', WORKSPACE / 'runs' / 'v3_determinism_report.json']
for path in required:
    print(path, 'exists=', path.exists())
    if not path.exists(): raise FileNotFoundError(f'Missing prerequisite: {path}')
print('All upstream artifacts are present.')


In [ ]:
# Run robustness variants.
import subprocess, sys
robustness = WORKSPACE / 'runs' / 'v3_robustness'
def run_stage(label, script_name, extra_args):
    cmd = [sys.executable, str(REPO / 'scripts' / script_name), *extra_args]
    print(f'\n=== {label} ===')
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:\n' + result.stderr)
    if result.returncode != 0: raise RuntimeError(f'{label} failed: {result.stderr or result.stdout}')
    return result
run_stage('robustness grid', 'run_v3_robustness.py', ['--data-root', str(DATA_ROOT), '--forecast-run', str(method), '--run-dir', str(robustness)])


In [ ]:
# Generate report tables and validate the locked method.
tables_run = WORKSPACE / 'runs' / 'v3_tables'
run_stage('report tables', 'generate_v3_tables.py', ['--baseline-run', str(baseline), '--portfolio-run', str(portfolio), '--method-run', str(method), '--ablation-run', str(ablation), '--run-dir', str(tables_run)])
run_stage('method validation', 'validate_v3_method.py', ['--data-root', str(DATA_ROOT), '--run-dir', str(method), '--report', str(WORKSPACE / 'runs' / 'v3_method_validation.json')])
print(list(tables_run.glob('*')))


In [ ]:
# Assemble and synchronize the final handoff.
final_handoff = WORKSPACE / 'runs' / 'v3_final_handoff'
run_stage('final handoff assembly', 'assemble_v3_final_handoff.py', ['--method-run', str(method), '--baseline-run', str(baseline), '--portfolio-run', str(portfolio), '--ablation-run', str(ablation), '--stats-run', str(stats), '--robustness-run', str(robustness), '--seed-run', str(WORKSPACE / 'runs' / 'v3_ptcst_seed_sweep'), '--tables-run', str(tables_run), '--leakage-report', str(WORKSPACE / 'runs' / 'v3_leakage_report.json'), '--determinism-report', str(WORKSPACE / 'runs' / 'v3_determinism_report.json'), '--risk-run', str(risk), '--cache-run', str(cache), '--output', str(final_handoff)])
DRIVE_HANDOFF = Path('/content/drive/MyDrive/kltn/runs/v3_final_handoff')
ARCHIVE = Path('/content/drive/MyDrive/kltn/archives') / 'v3_notebooks'
run_stage('Drive handoff sync', 'sync_v3_handoff.py', ['--source', str(final_handoff), '--destination', str(DRIVE_HANDOFF), '--notebook-root', str(REPO / 'notebooks'), '--archive-root', str(ARCHIVE)])
print('Final handoff:', DRIVE_HANDOFF)
